In [75]:
%pip install carla 
%pip install opencv-python
%pip install pygame

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [76]:
# Import the CARLA Python API library 
import carla
import math
import random
import time
import queue
import numpy as np
import cv2
import pygame
from PIL import Image
import json
import os

In [77]:
# Connect to the client and get the world object
client = carla.Client('localhost', 2000) 
client.set_timeout(200.0)
world  = client.reload_world()
bp_lib = world.get_blueprint_library()
spawn_points = world.get_map().get_spawn_points() 

In [78]:
# ##change weather to rainy
# weather = carla.WeatherParameters(
#     cloudiness=30.0,
#     precipitation=70.0,
#     sun_altitude_angle=70.0)

# world.set_weather(weather)

# print(world.get_weather())

In [79]:
#Set up the simulator in synchronous mode
settings = world.get_settings()
settings.synchronous_mode = True # Enables synchronous mode
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

38649

In [80]:
# Get the blueprint for the vehicle you want
vehicle_bp = bp_lib.find('vehicle.nissan.patrol_2021') 

# Try spawning the vehicle at a randomly chosen spawn point
vehicle = world.try_spawn_actor(vehicle_bp, random.choice(spawn_points))
spectator = world.get_spectator()

In [81]:
# Move the spectator behind the vehicle 
#spectator = world.get_spectator() 
#transform = carla.Transform(vehicle.get_transform().transform(carla.Location(x=-4,z=2.5)),vehicle.get_transform().rotation) 
#spectator.set_transform(transform) 

In [82]:
# Spawn vehicles
for i in range(50): 
    vehicle_bp = random.choice(bp_lib.filter('vehicle')) 
    
    npc = world.try_spawn_actor(vehicle_bp, random.choice(spawn_points)) 

In [83]:
# Set the all vehicles in motion using the Traffic Manager
#for v in world.get_actors().filter('*vehicle*'): 
#    v.set_autopilot(True) 

In [84]:
#camera on spectator (drone)
#drone = world.get_spectator()
#transform = drone.get_transform()
#location = transform.location
#rotation = transform.rotation
#Set 1 - Location(x=-57.660717, y=-71.480591, z=9.112691) Rotation(pitch=-8.841427, yaw=42.202526, roll=0.000119)
#Set 2 - Location(x=103.583794, y=133.276993, z=8.625797) Rotation(pitch=-13.471584, yaw=-178.655457, roll=0.000182)
#Set 3 - Location(x=-47.316780, y=146.866623, z=8.862714) Rotation(pitch=-11.895840, yaw=-120.825386, roll=0.000061)
#Location(x=-79.963348, y=-43.291283, z=10.184503) Rotation(pitch=-9.250794, yaw=-39.398540, roll=0.000071)
#Location(x=-7.185984, y=113.806297, z=15.188178) Rotation(pitch=-15.657955, yaw=167.534378, roll=0.000065)
#Location(x=-13.983668, y=108.808197, z=14.873873) Rotation(pitch=-14.254481, yaw=148.090759, roll=0.000044)
#location = carla.Location(x=14.0, y=29.0, z=6.0)
#rotation = carla.Rotation(pitch=0.0, yaw=math.pi/2, roll=0.0)
location = carla.Location(x=-13.983668, y=108.808197, z=14.873873) 
rotation = carla.Rotation(pitch=-14.254481, yaw=148.090759, roll=0.000044)
transform = carla.Transform(location, rotation)
# Set the spectator with an empty transform
#drone.set_transform(carla.Transform())

camera_drone = bp_lib.find('sensor.camera.rgb')
camera_drone.set_attribute('image_size_x', '1920')
camera_drone.set_attribute('image_size_y', '1080')
camera_drone.set_attribute('sensor_tick', '1.0')
camera_drone.set_attribute('fov', '90')
camera = world.spawn_actor(camera_drone, transform)


In [85]:
#Instance segmentation camera on drone
# drone = world.get_spectator()
# transform = drone.get_transform()
# location = transform.location
# rotation = transform.rotation
# # Set the spectator with an empty transform
# drone.set_transform(carla.Transform())

seg_camera = bp_lib.find('sensor.camera.instance_segmentation')
seg_camera.set_attribute('image_size_x', '1920')
seg_camera.set_attribute('image_size_y', '1080')
seg_camera.set_attribute('sensor_tick', '1.0')
seg_camera.set_attribute('fov', '90')
segmentation_camera = world.spawn_actor(seg_camera, transform)

In [86]:
# #semantic lidar on drone
# drone = world.get_spectator()
# transform = drone.get_transform()
# location = transform.location
# rotation = transform.rotation
# # Set the spectator with an empty transform
# drone.set_transform(carla.Transform())

sem_lidar = bp_lib.find('sensor.lidar.ray_cast_semantic')
sem_lidar.set_attribute('sensor_tick', '1.0')
lidar = world.spawn_actor(sem_lidar, transform)

In [87]:
#print lidar data
def semantic_lidar_data(point_cloud_data):
  for detection in point_cloud_data:
    print(detection)

In [88]:
#Camera geometric projection -> 3D points
def build_projection_matrix(w, h, fov, is_behind_camera=False):
    focal = w / (2.0 * np.tan(fov * np.pi / 360.0))
    K = np.identity(3)

    if is_behind_camera:
        K[0, 0] = K[1, 1] = -focal
    else:
        K[0, 0] = K[1, 1] = focal

    K[0, 2] = w / 2.0
    K[1, 2] = h / 2.0
    return K

In [89]:
#3D to 2D
def get_image_point(loc, K, w2c):
        # Calculate 2D projection of 3D coordinate

        # Format the input coordinate (loc is a carla.Position object)
        point = np.array([loc.x, loc.y, loc.z, 1])
        # transform to camera coordinates
        point_camera = np.dot(w2c, point)

        # New we must change from UE4's coordinate system to an "standard"
        # (x, y ,z) -> (y, -z, x)
        # and we remove the fourth componebonent also
        point_camera = [point_camera[1], -point_camera[2], point_camera[0]]

        # now project 3D->2D using the camera matrix
        point_img = np.dot(K, point_camera)
        # normalize
        point_img[0] /= point_img[2]
        point_img[1] /= point_img[2]

        return point_img[0:2]

In [90]:
# Get the world to camera matrix
world_2_camera = np.array(camera.get_transform().get_inverse_matrix())

# Get the attributes from the camera
image_w = camera_drone.get_attribute("image_size_x").as_int()
image_h = camera_drone.get_attribute("image_size_y").as_int()
fov = camera_drone.get_attribute("fov").as_float()

# Calculate the camera projection matrix to project from 3D -> 2D
K = build_projection_matrix(image_w, image_h, fov)
K_b = build_projection_matrix(image_w, image_h, fov, is_behind_camera=True)

In [91]:
image_queue = queue.Queue()
camera.listen(image_queue.put)

In [92]:
width = 1920
height = 1080
obj_id = 0

pygame.init()
window = pygame.display.set_mode(size=(width,height))
font = pygame.font.SysFont(None, 24)
video_surf = None

world.tick()
# Set all vehicles in motion using the Traffic Manager
for v in world.get_actors().filter('*vehicle*'): 
    v.set_autopilot(True)

simulation_dataset = {}
simulation_dataset["images"] = []
simulation_dataset["annotations"] = []
simulation_dataset["categories"] = [{"id": 1,"name": "car","supercategory": "car"}]

# Render 2D bboxes
running = True
tick = 1
max_tick = 2000
img_cnt = 0
while running:
    tick += 1
    print(tick)
    if tick > max_tick:
        break
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    # Retrieve and reshape the image
    world.tick()
    image = None
    try:
        image = image_queue.get(timeout=1)
        print("Received image!")
        img_cnt += 1
        print(f"Image_count: {img_cnt}")
    except:
        print("No image!")
        continue

    img = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))
    output_path = "dataset/stationary_camera/set6"
    output_dir = output_path+"/train"
    os.makedirs(output_dir, exist_ok=True)

    # Construct the full path for the image file
    img_filename = '%06d.png' % image.frame
    img_to_save = os.path.join(output_dir, img_filename)
    img_bgr = img[:, :, :3]
    img_rgb = img_bgr[:, :, ::-1]
    img_PIL = Image.fromarray(img_rgb)
    img_PIL.save(img_to_save)
    
    # Add image metadata to the dataset
    simulation_dataset["images"].append({
        "id": image.frame,
        "file_name": image.frame,
        "width": 1920,
        "height": 1080,
    })

    # Get the camera matrix 
    world_2_camera = np.array(camera.get_transform().get_inverse_matrix())

    for npc in world.get_actors().filter('*vehicle*'):
        bb = npc.bounding_box
        dist = npc.get_transform().location.distance(transform.location)

        # Filter for the vehicles within 50m
        if dist < 60:
            forward_vec = transform.get_forward_vector()
            ray = npc.get_transform().location - transform.location

            if forward_vec.dot(ray) > 0:
                verts = [v for v in bb.get_world_vertices(npc.get_transform())]
                x_max = -10000
                x_min = 10000
                y_max = -10000
                y_min = 10000

                for vert in verts:
                    p = get_image_point(vert, K, world_2_camera)
                    # Find the rightmost vertex
                    if p[0] > x_max:
                        x_max = p[0]
                    # Find the leftmost vertex
                    if p[0] < x_min:
                        x_min = p[0]
                    # Find the highest vertex
                    if p[1] > y_max:
                        y_max = p[1]
                    # Find the lowest  vertex
                    if p[1] < y_min:
                        y_min = p[1]

                # Check if the bbox is fully within image dimensions
                if 0 <= x_min < x_max <= width and 0 <= y_min < y_max <= height:
                    cv2.line(img, (int(x_min), int(y_min)), (int(x_max), int(y_min)), (0, 0, 255, 255), 1)
                    cv2.line(img, (int(x_min), int(y_max)), (int(x_max), int(y_max)), (0, 0, 255, 255), 1)
                    cv2.line(img, (int(x_min), int(y_min)), (int(x_min), int(y_max)), (0, 0, 255, 255), 1)
                    cv2.line(img, (int(x_max), int(y_min)), (int(x_max), int(y_max)), (0, 0, 255, 255), 1)

                    # Calculate width and height from x_max and y_max
                    box_width = x_max - x_min
                    box_height = y_max - y_min

                    # Update bbox to coco format
                    bbox = [x_min, y_min, box_width, box_height]
                    rounded_bbox = [round(value) for value in bbox]
                    obj_id += 1
                    area = round(box_width * box_height)

                    simulation_dataset["annotations"].append({
                        "id": obj_id,
                        "image_id": image.frame,
                        "category_id": 1,
                        "bbox": rounded_bbox,
                        "area": area,
                        "iscrowd": 0,
                        "segmentation": []
                    })
                    print(f"bbox : {rounded_bbox}")
                    # Render obj_id on top of the bbox
                    text_surface = font.render(str(obj_id), True, (255, 0, 0))  # Red color for the text
                    window.blit(text_surface, (x_min, y_min - 10))  # Display obj_id slightly above bbox

    video_surf = pygame.image.frombuffer(img, (image.width, image.height), "BGRA")
    window.blit(video_surf, (0, 0))
    pygame.display.flip()

pygame.quit()

# Save dataset annotations
output_json = output_path+"/annotations"
os.makedirs(output_json, exist_ok=True)
json_filename = os.path.join(output_dir, 'train.json')

with open(json_filename, 'w') as json_file:
    json.dump(simulation_dataset, json_file)

with open('simulation_data.json', 'w') as json_file:
    json.dump(simulation_dataset, json_file)


2
Received image!
Image_count: 1
bbox : [893, 540, 35, 38]
bbox : [839, 508, 89, 60]
3
No image!
4
No image!
5
No image!
6
No image!
7
No image!
8
No image!
9
No image!
10
No image!
11
No image!
12
No image!
13
No image!
14
No image!
15
No image!
16
No image!
17
No image!
18
No image!
19
No image!
20
No image!
21
Received image!
Image_count: 2
bbox : [893, 550, 35, 39]
bbox : [837, 517, 89, 60]
22
No image!
23
No image!
24
No image!
25
No image!
26
No image!
27
No image!
28
No image!
29
No image!
30
No image!
31
No image!
32
No image!
33
No image!
34
No image!
35
No image!
36
No image!
37
No image!
38
No image!
39
No image!
40
No image!
41
Received image!
Image_count: 3
bbox : [893, 550, 35, 39]
bbox : [799, 529, 96, 65]
42
No image!
43
No image!
44
No image!
45
No image!
46
No image!
47
No image!
48
No image!
49
No image!
50
No image!
51
No image!
52
No image!
53
No image!
54
No image!
55
No image!
56
No image!
57
No image!
58
No image!
59
No image!
60
No image!
61
Received image!
Ima

In [93]:
pygame.quit()

: 